In [2]:
#!pip install litellm

In [3]:
from litellm import completion
from google.colab import userdata
import requests
import json
import pandas as pd

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [ ]:
messages = [
    {"role": "system", "content": "Você é o chat da Terra e do Universo, que responde perguntas em português brasileiro sobre previsão do tempo na Terra e no espaço próximo à Terra, além de informações sobre terremotos."},
    {"role": "user", "content": "Qual é a frequência dos máximos solares?"}
]


In [ ]:
response = completion(
    model='groq/gemma2-9b-it',
    messages = messages,
    api_key=GROQ_API_KEY
)

print(response.choices[0].message.content)

Olá! 👋 Como Chat da Terra e do Universo, estou aqui para te ajudar com informações sobre o clima, a previsão do tempo e até sobre tremores da Terra. 🌎🪐

No que diz respeito ao Sol, os máximos solares ocorrem, em média, a cada 11 anos. ☀️ 

Mas lembre-se que esses intervalos nem sempre são perfeitos. Eles podem variar um pouco de um ciclo para o outro! 🧮  

Você tem outras perguntas sobre o Sol ou sobre o que está acontecendo nas estrelas? ✨ Me diga! 😄



In [ ]:
def call_groq_api(messages, model = 'groq/llama-3.3-70b-versatile'):
  response = completion(
    model=model,
    messages = messages,
    api_key=GROQ_API_KEY
  )
  return response.choices[0].message.content

In [ ]:
def chat():
  print('Iniciando o chat com modelo. Digite sair para encerrar')
  messages = [
      {"role": "system", "content": "Você é o chat da Terra e do Universo, que responde perguntas em português brasileiro sobre previsão do tempo na Terra e no espaço próximo à Terra, além de informações sobre terremotos."}
  ]

  while True:
    user_message = input('User: ')
    if user_message.lower() == 'sair':
      print('Saindo do chat')
      break

    messages.append({"role": "user", "content": user_message})
    model_response = call_groq_api(messages)
    messages.append({"role": "assistant", "content": model_response})
    print(f'Assistant: {model_response}')

In [ ]:
chat()

Iniciando o chat com modelo. Digite sair para encerrar
User: qual previsao do tempo para SP
Model: Olá! A previsão do tempo para São Paulo (SP) pode variar dependendo da época do ano e das condições climáticas atuais. No entanto, posso fornecer uma visão geral das condições climáticas típicas em São Paulo e uma previsão do tempo atualizada (lembre-se de que a previsão do tempo pode mudar rapidamente, então é sempre uma boa ideia verificar as fontes mais atualizadas).

**Clima em São Paulo:**
São Paulo tem um clima subtropical, com verões quentes e úmidos e invernos frios e secos. A temperatura média anual é de cerca de 22°C. A cidade experimenta uma estação seca durante os meses de maio a setembro e uma estação chuvosa durante os meses de outubro a abril.

**Previsão do tempo atual (verifique as fontes mais atualizadas):**
Como não tenho acesso a informações em tempo real, não posso fornecer uma previsão do tempo atualizada. No entanto, posso sugerir algumas fontes confiáveis para veri

KeyboardInterrupt: Interrupted by user

In [4]:
def previsao_tempo(city, country):
  WEATHER_API = userdata.get('WEATHER_API')

  url = f'http://api.openweathermap.org/data/2.5/weather?q={city},{country}&APPID={WEATHER_API}&lang=pt_br&units=metric'

  response = requests.get(url)
  data = response.json()

  return json.dumps(data)

previsao_tempo('São Paulo', 'Brasil')

'{"coord": {"lon": -46.6361, "lat": -23.5475}, "weather": [{"id": 802, "main": "Clouds", "description": "nuvens dispersas", "icon": "03n"}], "base": "stations", "main": {"temp": 18.94, "feels_like": 19.27, "temp_min": 18.2, "temp_max": 18.94, "pressure": 1021, "humidity": 91, "sea_level": 1021, "grnd_level": 929}, "visibility": 9000, "wind": {"speed": 1.54, "deg": 160}, "clouds": {"all": 40}, "dt": 1747904868, "sys": {"type": 2, "id": 2082654, "country": "BR", "sunrise": 1747906612, "sunset": 1747945794}, "timezone": -10800, "id": 3448439, "name": "S\\u00e3o Paulo", "cod": 200}'

### Adicionando função

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "previsao_tempo",
            "description": "Retorna a previsão do tempo para uma cidade e país específicos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nome da cidade para consultar a previsão."
                    },
                    "country": {
                        "type": "string",
                        "description": "Sigla do país (ex: 'BR' para Brasil)."
                    }
                },
                "required": ["city", "country"]
            }
        }
    }
]

In [ ]:
def call_groq_api(messages, model = 'groq/llama-3.3-70b-versatile'):
  global tools
  response = completion(
    model=model,
    messages = messages,
    tools = tools,
    tool_choice = 'auto',
    api_key=GROQ_API_KEY
  )

  resposta_text =  response.choices[0].message
  chamada_ferramentas = resposta_text.tool_calls

  if chamada_ferramentas:
    available_func = {
        'previsao_tempo': previsao_tempo
    }
    for tool_call in chamada_ferramentas:
      function_name = tool_call.function.name
      function_to_call = available_func[function_name]
      function_args = json.loads(tool_call.function.arguments)

      function_response = function_to_call(
          city = function_args.get('city'),
          country = function_args.get('country')
      )

      return function_response

  else:
    return resposta_text.content

In [ ]:
chat()

Iniciando o chat com modelo. Digite sair para encerrar
User: como vai?
Assistant: Estou funcionando corretamente, obrigado por perguntar! Estou aqui para ajudar com qualquer pergunta sobre previsão do tempo, terremotos ou outras questões relacionadas à Terra e ao espaço próximo à Terra. Como posso ajudar você hoje?
User: sao paulo


KeyError: 'previsao do tempo'

### Adicionando segunda função

In [43]:
def verificar_tempestade_solar():
  url = "https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json"
  response = requests.get(url)
  if response.status_code == 200:
    data = response.json()
    latest_kp = float(data[-1][1])
    if latest_kp >= 5:
      return f'Alerta de tempestade solar. Kp atual: {latest_kp}'
    else:
      return f'Sem alertas de tempestade solar. Kp atual: {latest_kp}'
  else:
    return 'Erro ao obter dados do NOAA'

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "previsao_tempo",
            "description": "Retorna a previsão do tempo para uma cidade e país específicos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nome da cidade para consultar a previsão."
                    },
                    "country": {
                        "type": "string",
                        "description": "Sigla do país (ex: 'BR' para Brasil)."
                    }
                },
                "required": ["city", "country"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "verificar_tempestade_solar",
            "description": "Verifica se há tempestade solar em andamento",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

In [ ]:
def call_groq_api(messages, model = 'groq/llama-3.3-70b-versatile'):
  global tools
  response = completion(
    model=model,
    messages = messages,
    tools = tools,
    tool_choice = 'auto',
    api_key=GROQ_API_KEY
  )

  resposta_text =  response.choices[0].message
  chamada_ferramentas = resposta_text.tool_calls

  if chamada_ferramentas:
    available_func = {
        'previsao_tempo': previsao_tempo,
        'verificar_tempestade_solar': verificar_tempestade_solar
    }
    for tool_call in chamada_ferramentas:
      function_name = tool_call.function.name
      function_to_call = available_func[function_name]
      function_args = json.loads(tool_call.function.arguments)

      match function_name:
        case 'previsao_tempo':
          function_response = function_to_call(
              city = function_args.get('city'),
              country = function_args.get('country')
          )

        case 'verificar_tempestade_solar':
          function_response = function_to_call()

      return function_response

  else:
    return resposta_text.content

In [ ]:
chat()

Iniciando o chat com modelo. Digite sair para encerrar
User: sair
Saindo do chat


### Adicionando mais uma função

In [5]:
def extrair_sismos():
    url = 'https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_day.csv'

    df = pd.read_csv(url)
    return df

extrair_sismos()

,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-05-22T08:34:36.889Z,-37.4615,51.4073,10.000,5.7,mww,60,26,8.968,0.67,...,2025-05-22T08:59:41.693Z,South Indian Ocean,earthquake,10.55,1.842,0.093,11,reviewed,us,us
1,2025-05-22T07:50:38.464Z,-37.4080,51.5554,10.000,5.3,mww,49,54,9.018,0.62,...,2025-05-22T08:36:31.040Z,South Indian Ocean,earthquake,11.48,1.786,0.110,8,reviewed,us,us
2,2025-05-22T07:01:14.340Z,-22.7791,-174.8615,10.000,4.9,mb,34,139,5.902,0.87,...,2025-05-22T07:18:35.040Z,"159 km S of ‘Ohonua, Tonga",earthquake,13.59,1.892,0.112,25,reviewed,us,us
3,2025-05-22T06:51:31.206Z,19.0205,-102.0748,108.838,4.7,mb,104,171,1.014,0.79,...,2025-05-22T07:20:42.725Z,"1 km ESE of Nueva Italia de Ruiz, Mexico",earthquake,9.97,7.637,0.024,520,reviewed,us,us
4,2025-05-22T06:40:39.142Z,77.9980,13.7956,10.000,4.5,mb,58,90,0.996,0.71,...,2025-05-22T06:55:21.040Z,"49 km WSW of Longyearbyen, Svalbard and Jan Mayen",earthquake,6.91,1.898,0.076,51,reviewed,us,us
5,2025-05-22T05:41:39.522Z,-4.0572,143.0112,88.762,4.6,mb,52,77,2.767,0.73,...,2025-05-22T06:00:04.040Z,"27 km NE of Ambunti, Papua New Guinea",earthquake,7.90,7.692,0.080,46,reviewed,us,us
6,2025-05-22T03:19:35.341Z,35.8112,25.8577,64.000,6.2,mww,142,35,0.946,0.95,...,2025-05-22T09:01:20.433Z,"61 km NNE of Eloúnda, Greece",earthquake,5.60,1.870,0.036,75,reviewed,us,us
7,2025-05-22T00:47:47.824Z,-39.3033,179.9208,12.660,4.5,mb,21,161,2.151,0.63,...,2025-05-22T01:07:39.040Z,"174 km ESE of Wainui, New Zealand",earthquake,9.02,6.215,0.191,8,reviewed,us,us
8,2025-05-22T00:25:43.716Z,-55.9417,-26.6532,45.367,5.2,mb,58,68,5.884,0.80,...,2025-05-22T00:42:45.040Z,South Sandwich Islands region,earthquake,10.92,7.224,0.060,93,reviewed,us,us
9,2025-05-21T20:11:11.912Z,17.2273,147.1652,35.000,4.7,mb,111,133,2.377,0.80,...,2025-05-21T20:36:06.040Z,"269 km NE of Saipan, Northern Mariana Islands",earthquake,9.66,1.917,0.048,130,reviewed,us,us


In [41]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "previsao_tempo",
            "description": "Retorna a previsão do tempo para uma cidade e país específicos.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "Nome da cidade para consultar a previsão."
                    },
                    "country": {
                        "type": "string",
                        "description": "Sigla do país (ex: 'BR' para Brasil)."
                    }
                },
                "required": ["city", "country"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "verificar_tempestade_solar",
            "description": "Verifica se há tempestade solar em andamento",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "extrair_sismos",
            "description": "Extrai dados de sismos da USGS",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]

In [42]:
def call_groq_api(messages, model = 'groq/llama-3.3-70b-versatile'):
  global tools
  response = completion(
    model=model,
    messages = messages,
    tools = tools,
    tool_choice = 'auto',
    api_key=GROQ_API_KEY
  )

  resposta_text =  response.choices[0].message
  chamada_ferramentas = resposta_text.tool_calls

  if chamada_ferramentas:
    available_func = {
        'previsao_tempo': previsao_tempo,
        'verificar_tempestade_solar': verificar_tempestade_solar,
        'extrair_sismos': extrair_sismos
    }
    for tool_call in chamada_ferramentas:
      function_name = tool_call.function.name
      function_to_call = available_func[function_name]
      function_args = json.loads(tool_call.function.arguments)

      match function_name:
        case 'previsao_tempo':
          function_response = function_to_call(
              city = function_args.get('city'),
              country = function_args.get('country')
          )

        case 'verificar_tempestade_solar':
          function_response = function_to_call()

        case 'extrair_sismos':
          function_response = function_to_call()

      return function_response

  else:
    return resposta_text.content

In [8]:
def chat():
    print("Iniciando chat com o modelo. Digite 'sair' para encerrar.")

    # Histórico de mensagens
    messages = [{"role": "system", "content": """
    Você é o Chat da Terra e do Universo e responde em português brasileiro
    perguntas sobre a previsão do tempo na Terra e do espaço próximo à Terra, além de informações sobre terremotos.
    """}]

    while True:
        user_message = input("Você: ")
        if user_message.lower() == "sair":
            print("Encerrando chat. Até a próxima!")
            break

        # Adicionar a mensagem do usuário ao histórico
        messages.append({"role": "user", "content": user_message})

        # Chamar a API com o histórico completo
        model_response = call_groq_api(messages)

        # Exibir a resposta do assistente
        display(model_response)
        if isinstance(model_response, pd.DataFrame):
            print("O model_response é um DataFrame.")
            texto_corrido = ""
            for index, row in model_response.iterrows():
                texto_corrido += f"Evento {index + 1}: Magnitude {row['mag']}, Local {row['place']}, Tempo {row['time']}\n"

            model_response = texto_corrido


        # Adicionar a resposta do modelo ao histórico
        messages.append({"role": "assistant", "content": model_response})

In [ ]:
chat()

Iniciando chat com o modelo. Digite 'sair' para encerrar.
Você: tevemuitos sismos?


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-05-21T06:34:13.789Z,-30.6338,-178.7672,131.007,4.7,mb,32,102,1.564,1.09,...,2025-05-21T07:18:12.040Z,"Kermadec Islands, New Zealand",earthquake,13.20,8.601,0.083,45,reviewed,us,us
1,2025-05-21T05:32:20.988Z,-9.2164,123.7709,107.609,4.5,mb,24,79,0.725,0.80,...,2025-05-21T06:44:33.040Z,"42 km N of Naisano Dua, Indonesia",earthquake,8.81,6.599,0.124,19,reviewed,us,us
2,2025-05-21T00:38:12.265Z,-56.9630,-67.1157,10.000,5.5,mb,85,97,4.436,0.68,...,2025-05-21T03:11:00.592Z,"251 km SSE of Ushuaia, Argentina",earthquake,11.73,1.754,0.038,238,reviewed,us,us
3,2025-05-20T18:44:38.822Z,-56.7812,-67.3921,10.000,4.6,mb,25,184,3.423,0.43,...,2025-05-20T19:59:40.040Z,"226 km SSE of Ushuaia, Argentina",earthquake,16.19,1.957,0.122,20,reviewed,us,us
4,2025-05-20T18:40:13.467Z,-21.0640,-179.2534,615.474,4.6,mb,44,125,4.167,0.95,...,2025-05-20T19:19:35.040Z,Fiji region,earthquake,14.14,9.009,0.088,39,reviewed,us,us
5,2025-05-20T17:57:03.317Z,-17.3128,66.2824,10.000,4.9,mb,61,132,18.185,0.69,...,2025-05-20T18:40:33.040Z,Mauritius - Reunion region,earthquake,14.02,1.882,0.082,47,reviewed,us,us
6,2025-05-20T16:22:49.369Z,0.6994,-25.9792,10.000,4.9,mb,68,101,11.839,0.61,...,2025-05-20T16:52:30.040Z,central Mid-Atlantic Ridge,earthquake,7.76,1.891,0.071,62,reviewed,us,us
7,2025-05-20T15:05:58.710Z,-3.8503,144.7662,10.000,6.4,mww,152,50,3.156,1.19,...,2025-05-21T04:21:45.081Z,"81 km ENE of Angoram, Papua New Guinea",earthquake,8.18,1.874,0.052,35,reviewed,us,us
8,2025-05-20T12:53:01.218Z,-23.3039,-179.9541,539.507,4.7,mb,79,57,5.837,1.05,...,2025-05-20T13:13:45.040Z,south of the Fiji Islands,earthquake,11.13,6.740,0.033,282,reviewed,us,us


O model_response é um DataFrame.
Você: qual de maior magnitude?


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2025-05-21T06:34:13.789Z,-30.6338,-178.7672,131.007,4.7,mb,32,102,1.564,1.09,...,2025-05-21T07:18:12.040Z,"Kermadec Islands, New Zealand",earthquake,13.20,8.601,0.083,45,reviewed,us,us
1,2025-05-21T05:32:20.988Z,-9.2164,123.7709,107.609,4.5,mb,24,79,0.725,0.80,...,2025-05-21T06:44:33.040Z,"42 km N of Naisano Dua, Indonesia",earthquake,8.81,6.599,0.124,19,reviewed,us,us
2,2025-05-21T00:38:12.265Z,-56.9630,-67.1157,10.000,5.5,mb,85,97,4.436,0.68,...,2025-05-21T03:11:00.592Z,"251 km SSE of Ushuaia, Argentina",earthquake,11.73,1.754,0.038,238,reviewed,us,us
3,2025-05-20T18:44:38.822Z,-56.7812,-67.3921,10.000,4.6,mb,25,184,3.423,0.43,...,2025-05-20T19:59:40.040Z,"226 km SSE of Ushuaia, Argentina",earthquake,16.19,1.957,0.122,20,reviewed,us,us
4,2025-05-20T18:40:13.467Z,-21.0640,-179.2534,615.474,4.6,mb,44,125,4.167,0.95,...,2025-05-20T19:19:35.040Z,Fiji region,earthquake,14.14,9.009,0.088,39,reviewed,us,us
5,2025-05-20T17:57:03.317Z,-17.3128,66.2824,10.000,4.9,mb,61,132,18.185,0.69,...,2025-05-20T18:40:33.040Z,Mauritius - Reunion region,earthquake,14.02,1.882,0.082,47,reviewed,us,us
6,2025-05-20T16:22:49.369Z,0.6994,-25.9792,10.000,4.9,mb,68,101,11.839,0.61,...,2025-05-20T16:52:30.040Z,central Mid-Atlantic Ridge,earthquake,7.76,1.891,0.071,62,reviewed,us,us
7,2025-05-20T15:05:58.710Z,-3.8503,144.7662,10.000,6.4,mww,152,50,3.156,1.19,...,2025-05-21T04:21:45.081Z,"81 km ENE of Angoram, Papua New Guinea",earthquake,8.18,1.874,0.052,35,reviewed,us,us
8,2025-05-20T12:53:01.218Z,-23.3039,-179.9541,539.507,4.7,mb,79,57,5.837,1.05,...,2025-05-20T13:13:45.040Z,south of the Fiji Islands,earthquake,11.13,6.740,0.033,282,reviewed,us,us


O model_response é um DataFrame.
Você: qual é o maior mag?


...

Você: vc sabe programar python?


'{"coord": {"lon": -43.2075, "lat": -22.9028}, "weather": [{"id": 800, "main": "Clear", "description": "c\\u00e9u limpo", "icon": "01d"}], "base": "stations", "main": {"temp": 20.72, "feels_like": 21.01, "temp_min": 17.78, "temp_max": 20.98, "pressure": 1017, "humidity": 83, "sea_level": 1017, "grnd_level": 1018}, "visibility": 10000, "wind": {"speed": 3.09, "deg": 360}, "clouds": {"all": 0}, "dt": 1747820425, "sys": {"type": 2, "id": 2098643, "country": "BR", "sunrise": 1747819294, "sunset": 1747858657}, "timezone": -10800, "id": 3451190, "name": "Rio de Janeiro", "cod": 200}'

Você: vc sabe programar python? nao chama nenhuma funçao


'Sim, eu sei programar em Python! Embora eu seja um modelo de linguagem treinado por máquina, eu tenho conhecimento sobre a linguagem Python e posso entender e gerar código Python. No entanto, é importante notar que eu não tenho a capacidade de executar código Python ou ter acesso a recursos externos como bancos de dados ou sistemas de arquivos. Minha habilidade se limita a gerar código Python baseado em minha compreensão da linguagem e responder a perguntas relacionadas a programação em Python.'

Você: sair
Encerrando chat. Até a próxima!


In [10]:
#!pip install pinecone

In [11]:
PINECONE_API_KEY = userdata.get('PINECONE_API')

from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)

In [12]:
data = [
    {
        "id": "occurrence1",
        "text": "Ouro presente em veios de quartzo em uma formação hidrotermal. Observa-se alta concentração de ouro em zonas de fraturas e falhas."
    },
    {
        "id": "occurrence2",
        "text": "Associação do ouro com sulfetos, especialmente pirita e arsenopirita, em ambiente de rochas metavulcânicas. Indica potencial para depósito orogênico de ouro."
    },
    {
        "id": "occurrence3",
        "text": "Ouro aluvial encontrado em depósitos de cascalho próximo a rios e córregos. Indica transporte e concentração secundária de ouro."
    },
    {
        "id": "occurrence4",
        "text": "Presença de ouro em formações de skarn associadas a intrusões ígneas graníticas. Indica formação relacionada a processos de metamorfismo de contato."
    },
    {
        "id": "occurrence5",
        "text": "Ouro disseminado em formações sedimentares de origem marinha, em conglomerados ricos em minerais pesados. Potencial para depósito do tipo placer ou paleoplacer."
    },
]

In [13]:
index = pc.Index('geologia')

In [14]:
embeddings = pc.inference.embed(
    'multilingual-e5-large',
    inputs=[d['text'] for d in data],
    parameters={'input_type': 'passage'}
)

In [15]:
vectors = []

for d, e in zip(data, embeddings):
  vectors.append({
      'id': d['id'],
      'values': e['values'],
      'metadata': {'text': d['text']}
  })

In [ ]:
vectors

[{'id': 'occurrence1',
  'values': [0.01340484619140625,
   0.007171630859375,
   -0.03558349609375,
   -0.060943603515625,
   0.017730712890625,
   -0.040283203125,
   -0.03375244140625,
   0.11639404296875,
   0.04986572265625,
   -0.0144805908203125,
   0.055389404296875,
   0.018341064453125,
   -0.02496337890625,
   0.0059661865234375,
   -0.0005908012390136719,
   -0.00984954833984375,
   -0.0242462158203125,
   0.03631591796875,
   0.00708770751953125,
   0.0012540817260742188,
   0.034912109375,
   0.007495880126953125,
   -0.042144775390625,
   -0.04339599609375,
   -0.039794921875,
   -0.01369476318359375,
   -0.05267333984375,
   -0.0181427001953125,
   -0.0277862548828125,
   -0.02227783203125,
   -0.00814056396484375,
   0.023651123046875,
   -0.04278564453125,
   -0.037506103515625,
   -0.032470703125,
   0.04693603515625,
   0.07269287109375,
   0.046844482421875,
   -0.035614013671875,
   0.05657958984375,
   -0.0325927734375,
   0.051727294921875,
   -0.016769409179687

In [16]:
index.upsert(vectors=vectors, namespace='ns1')

{'upserted_count': 5}

In [ ]:
query = 'o ouro é de sendimentos da marinha?'

x = pc.inference.embed(
    'multilingual-e5-large',
    inputs=[query],
    parameters={'input_type': 'query'}
)

In [ ]:
results = index.query(
    namespace='ns1',
    top_k=3,
    vector=x[0].values,
    include_metadata=True,
    include_values=False
)

results

{'matches': [{'id': 'occurrence5',
              'metadata': {'text': 'Ouro disseminado em formações sedimentares '
                                   'de origem marinha, em conglomerados ricos '
                                   'em minerais pesados. Potencial para '
                                   'depósito do tipo placer ou paleoplacer.'},
              'score': 0.875868559,
              'values': []},
             {'id': 'occurrence1',
              'metadata': {'text': 'Ouro presente em veios de quartzo em uma '
                                   'formação hidrotermal. Observa-se alta '
                                   'concentração de ouro em zonas de fraturas '
                                   'e falhas.'},
              'score': 0.843479931,
              'values': []},
             {'id': 'occurrence3',
              'metadata': {'text': 'Ouro aluvial encontrado em depósitos de '
                                   'cascalho próximo a rios e córregos. Indica '
        

### Adicionando livro como RAG

In [17]:
#!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 kB 16.2 MB/s eta 0:00:00


In [18]:
import pypdf

In [19]:
def chunk_text(text, chunk_size=500):
  #divide o texto em partes menores
  return[text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

In [21]:
with open('livro.pdf', 'rb') as f:
  reader = pypdf.PdfReader(f)
  text = ''

  for page_num in range(len(reader.pages)):
    page = reader.pages[page_num]
    text += page.extract_text()

In [22]:
chunks = chunk_text(text)

In [23]:
chunks[100]

'os do \ncampo de estudo discutido. \nJuntos com estes textos, esta o disponí -\nveis outros com enfoque mais geolo gico, \ncom descriço es de minerais, rochas, so-\nlos e estruturas. Um deles, que deve ser \nrecomendado, e  o de R.E. Goodman \n(1993, Engineering Geology: Rocks in En-\ngineering Construction ), que trata os \ngrandes grupos de rochas separadamen-\nte, reconhecendo que apresentam pro-\npriedades especí ficas e evoluço es distin-\ntas (por ex., no quesito alteração). A.C. \nMcLean e C.D. Gribbl'

In [24]:
data = []

#plano free da pinecone necessario menos tokens
for i, chunk in enumerate(chunks[:90], start=1):
  data.append({
      'id': f'chunk{i}',
      'text': chunk.strip()
  })

for chunk in data[:5]:
  print(chunk)

{'id': 'chunk1', 'text': 'Universidade de São Paulo \nInstituto de Geociências \nGeologia básica para engenheiros \nHorstpeter H. G. J. Ulbrich \nJose  B. de Madureira Filho \nEliane A. Del Lama \nLauro K. Dehira \nISBN:  978-65-86403-05-3 \nDOI:10.11606/9786586403053 \nSa o Paulo \n2023 Esta obra é de acesso aberto.  É permitida a reprodução parcial ou total desta obra, desde que citada a fonte e autoria, proibin do qualquer uso para fins \ncomerciais e respeitando a Licença Creative Commons indicada: Licença Creative Commons (CC'}
{'id': 'chunk2', 'text': 'BY-NC-ND 4.0) Atribuição-Não Comercial-Sem Deriva-\nções 4.0 Internacional.  \nPara mais informações, leia a política de privacidade do Portal de Livros Abertos da Agência de Bibliotecas e Coleções Digitais, Universidade de \nSão Paulo, SP, Brasil.   \n \n \nUniversidade de São Paulo \nReitor: Prof. Dr. Carlos Gilberto Carlotti Junior \nVice-Reitora: Profa. Dra. Maria Arminda do Nascimento Arruda \nInstituto de Geociências \nDireto

In [25]:
embeddings = pc.inference.embed(
    "multilingual-e5-large",
    inputs=[d['text'] for d in data],
    parameters={
        "input_type": "passage"
    }
)

In [26]:
vectors = []
for d, e in zip(data, embeddings):
    vectors.append({
        "id": d['id'],
        "values": e['values'],
        "metadata": {'text': d['text']}
    })

index.upsert(
    vectors=vectors,
    namespace="ns1"
)

{'upserted_count': 90}

In [27]:
def info_geologia(query):
  x = pc.inference.embed(
      'multilingual-e5-large',
      inputs=[query],
      parameters={'input_type': 'query'}
  )
  results = index.query(
      namespace='ns1',
      top_k=3,
      vector=x[0].values,
      include_metadata=True,
      include_values=False
  )
  return results

In [28]:
query = 'o ouro ocorre em sedimentos de origem marinha?'
resposta = info_geologia(query)
resposta.matches[0].metadata['text']

'Ouro disseminado em formações sedimentares de origem marinha, em conglomerados ricos em minerais pesados. Potencial para depósito do tipo placer ou paleoplacer.'

In [29]:
# Função para iniciar o chat, mantendo o histórico
def chat():
    print("Iniciando chat com o modelo. Digite 'sair' para encerrar.")

    # Histórico de mensagens
    messages = [{"role": "system", "content": """
    Você é o Chat da Terra e do Universo e responde em português brasileiro
    perguntas sobre a previsão do tempo na Terra e do espaço próximo à Terra, além de informações sobre terremotos.
    """}]

    while True:
        user_message = input("Você: ")
        if user_message.lower() == "sair":
            print("Encerrando chat. Até a próxima!")
            break

        # Adicionar a mensagem do usuário ao histórico
        messages.append({"role": "user", "content": user_message})

        # Verificar se o tema é geologia
        if "geologia" in user_message.lower():
            # Chamar a função info_geologia para obter informações adicionais
            resposta = info_geologia(user_message)

            # Adicionar as informações de geologia ao histórico como contexto extra
            geologia_info = f"Informações adicionais sobre geologia: {resposta.matches[0].metadata['text']}"
            messages.append({"role": "system", "content": geologia_info})

        # Chamar a API com o histórico completo
        model_response = call_groq_api(messages)

        # Exibir a resposta do assistente
        display(model_response)
        if isinstance(model_response, pd.DataFrame):
            print("O model_response é um DataFrame.")
            texto_corrido = ""
            for index, row in model_response.iterrows():
                texto_corrido += f"Evento {index + 1}: Magnitude {row['mag']}, Local {row['place']}, Tempo {row['time']}\n"

            model_response = texto_corrido

        # Adicionar a resposta do modelo ao histórico
        messages.append({"role": "assistant", "content": model_response})

In [ ]:
chat()

Iniciando chat com o modelo. Digite 'sair' para encerrar.
Você: vai chover em sp?


'{"coord": {"lon": -46.6361, "lat": -23.5475}, "weather": [{"id": 803, "main": "Clouds", "description": "nublado", "icon": "04n"}], "base": "stations", "main": {"temp": 20.7, "feels_like": 21.02, "temp_min": 20.2, "temp_max": 21.36, "pressure": 1019, "humidity": 84, "sea_level": 1019, "grnd_level": 929}, "visibility": 10000, "wind": {"speed": 4.12, "deg": 150}, "clouds": {"all": 75}, "dt": 1747868552, "sys": {"type": 1, "id": 8394, "country": "BR", "sunrise": 1747820184, "sunset": 1747859412}, "timezone": -10800, "id": 3448439, "name": "S\\u00e3o Paulo", "cod": 200}'

Você: quantos anos vc tem


'{"coord": {"lon": -46.6361, "lat": -23.5475}, "weather": [{"id": 803, "main": "Clouds", "description": "nublado", "icon": "04n"}], "base": "stations", "main": {"temp": 20.67, "feels_like": 20.99, "temp_min": 20.2, "temp_max": 21.16, "pressure": 1019, "humidity": 84, "sea_level": 1019, "grnd_level": 929}, "visibility": 10000, "wind": {"speed": 4.12, "deg": 150}, "clouds": {"all": 75}, "dt": 1747868701, "sys": {"type": 1, "id": 8394, "country": "BR", "sunrise": 1747820184, "sunset": 1747859412}, "timezone": -10800, "id": 3448439, "name": "S\\u00e3o Paulo", "cod": 200}'

Você: ouro ocorre quando?


...

Você: ouro ocorre quando?


'{"coord": {"lon": -46.6361, "lat": -23.5475}, "weather": [{"id": 803, "main": "Clouds", "description": "nublado", "icon": "04n"}], "base": "stations", "main": {"temp": 20.67, "feels_like": 20.99, "temp_min": 20.2, "temp_max": 21.16, "pressure": 1019, "humidity": 84, "sea_level": 1019, "grnd_level": 929}, "visibility": 10000, "wind": {"speed": 4.12, "deg": 150}, "clouds": {"all": 75}, "dt": 1747868701, "sys": {"type": 1, "id": 8394, "country": "BR", "sunrise": 1747820184, "sunset": 1747859412}, "timezone": -10800, "id": 3448439, "name": "S\\u00e3o Paulo", "cod": 200}'

Você: ouro ocorre quando? sai da função previsao tempo


'Não estou mais na função previsão do tempo. O ouro ocorre quando há uma combinação de condições atmosféricas que fazem com que a luz do sol seja refratada e dispersa, criando o efeito visual de um arco-íris com apenas uma cor, o ouro. Isso geralmente ocorre após uma chuva ou quando há gotículas de água suspensas no ar.'

Você: sair
Encerrando chat. Até a próxima!


### Disponibilizando o chat no gradio

In [31]:
#!pip install gradio

In [45]:
def response(message, history):
    messages = [{"role": "system", "content": """
    Você é o Chat da Terra e do Universo e responde em português brasileiro
    perguntas sobre a previsão do tempo na Terra e do espaço próximo à Terra, além de informações sobre terremotos.
    """}]

    # Adicionar o histórico anterior ao histórico de mensagens
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})

    # Adicionar a nova mensagem do usuário
    messages.append({"role": "user", "content": message})

    # Verificar se o tema é geologia
    if "geologia" in message.lower():
        resposta = info_geologia(message)
        geologia_info = f"Informações adicionais sobre geologia: {resposta.matches[0].metadata['text']}"
        messages.append({"role": "system", "content": geologia_info})

    # Obter a resposta do modelo
    model_response = call_groq_api(messages)

    # Se a resposta for um DataFrame, convertê-la para texto
    if isinstance(model_response, pd.DataFrame):
        texto_corrido = ""
        for index, row in model_response.iterrows():
            texto_corrido += f"Evento {index + 1}: Magnitude {row['mag']}, Local {row['place']}, Tempo {row['time']}\n"
        model_response = texto_corrido

    # Retornar a resposta como string para Gradio
    return model_response

In [33]:
import gradio as gr

In [ ]:
gr.ChatInterface(
    response,
    title='¬Chat da terra¬',
    textbox= gr.Textbox(placeholder="Digite sua msg aqui..."),
    type='messages'
).launch(debug=True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://178a6f310847b0456c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://178a6f310847b0456c.gradio.live


In [34]:
from transformers import pipeline
import numpy as np

In [36]:
#necessário o HK_TOKEN para ler o modelo de descrição de áudio

transcritor = pipeline('automatic-speech-recognition',
                       model='openai/whisper-base',
                       generate_kwargs={"task":"transcribe", "language":"<|pt|>"})

def transcricao_audio(audio):
  sr, y = audio
  if y.ndim > 1:
    y = y.mean(axis=1)

  y = y.astype(np.float32)
  y /= np.max(np.abs(y))

  return transcritor({'sampling_rate': sr, 'raw': y})['text']

Device set to use cpu


In [46]:
with gr.Blocks() as demo:
    with gr.Tab("Chat da Terra e do Universo"):
        gr.ChatInterface(
            response,
            title='🌍☀️🌧️ Chat da Terra e do Universo',
            textbox=gr.Textbox(placeholder="Digite sua mensagem aqui..."),
            submit_btn=gr.Button("Enviar")

    )

    with gr.Tab("Assistente de áudio"):
        gr.Interface(
            transcricao_audio,
            gr.Audio(sources="microphone"),
            "text",
        )
demo.launch(debug=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://f01c3a42e135dbf234.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://f01c3a42e135dbf234.gradio.live


In [47]:
def responde_audio(audio):
    messages = [{"role": "system", "content": """
    Você é o Chat da Terra e do Universo e responde em português brasileiro
    perguntas sobre a previsão do tempo na Terra e do espaço próximo à Terra, além de informações sobre terremotos.
    """}]
    message = transcricao_audio(audio)


    # Adicionar a nova mensagem do usuário
    messages.append({"role": "user", "content": message})

    # Verificar se o tema é geologia
    if "geologia" in message.lower():
        resposta = info_geologia(message)
        geologia_info = f"Informações adicionais sobre geologia: {resposta.matches[0].metadata['text']}"
        messages.append({"role": "system", "content": geologia_info})

    # Obter a resposta do modelo
    model_response = call_groq_api(messages)

    # Se a resposta for um DataFrame, convertê-la para texto
    if isinstance(model_response, pd.DataFrame):
        texto_corrido = ""
        for index, row in model_response.iterrows():
            texto_corrido += f"Evento {index + 1}: Magnitude {row['mag']}, Local {row['place']}, Tempo {row['time']}\n"
            model_response = texto_corrido

    # Retornar a resposta como string para Gradio
    return model_response

In [50]:
with gr.Blocks() as demo:
    with gr.Tab("Chat da Terra e do Universo"):
        gr.ChatInterface(
            response,
            title='Chat da Terra e do Universo',
            textbox=gr.Textbox(placeholder="Digite sua mensagem aqui..."),
            submit_btn=gr.Button("Enviar")
    )

    with gr.Tab("Assistente de áudio"):
      gr.Interface(
          fn=responde_audio,
          inputs=gr.Audio(sources="microphone"),
          outputs="text",
          title="Assistente de áudio"
      )

demo.launch(debug=True)

/usr/local/lib/python3.11/dist-packages/gradio/chat_interface.py:339: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d07e11fa4574596531.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7862 <> https://d07e11fa4574596531.gradio.live


- **gTTS (Google Text-to-Speech)**

A gTTS é uma das bibliotecas mais populares para transformar texto em voz, usando a API do Google Translate. Ela é leve, fácil de instalar e oferece suporte a uma variedade de idiomas. A grande vantagem da gTTS é que é possível criar arquivos de áudio com apenas algumas linhas de código.

Exemplo:


```
pip install gtts

from gtts import gTTS
import os

# Texto que queremos converter em áudio
texto = "Olá! Este é um exemplo de conversão de texto para voz usando a biblioteca gTTS."

# Configuração da gTTS (idioma: português)
tts = gTTS(text=texto, lang='pt', slow=False)

# Salvando o arquivo de áudio
tts.save("exemplo_audio.mp3")

# Reproduzindo o áudio (opcional)
os.system("start exemplo_audio.mp3")  # No Windows, use "start"; no Mac, "afplay"; no Linux, "mpg321"
```

- **pyttsx3**

A biblioteca pyttsx3 permite a conversão de texto em voz offline, sem necessidade de conexão com a internet. Ela é uma ótima alternativa para quem precisa de uma solução local. Além disso, o pyttsx3 permite o controle da velocidade da fala e do volume. Com o pyttsx3, você tem um controle maior sobre a voz, e ele é ideal para ambientes que exigem TTS offline.

- **Amazon Polly**

A Amazon Polly é um serviço de texto para voz da AWS, capaz de gerar vozes naturais com excelente qualidade. Essa solução é paga, mas oferece vozes realistas e configurações avançadas, como controle de tom e pausas. Ela é amplamente utilizada em aplicações que demandam vozes mais naturais e com uma variedade maior de idiomas e sotaques. Para usar o Polly, é necessário ter uma conta AWS com permissões apropriadas para acessar o serviço.

- **Fala de IA do Azure**

A Fala de IA do Azure oferece um serviço robusto de TTS, suportando uma vasta gama de idiomas e permitindo configurações avançadas para gerar vozes naturais. É uma alternativa poderosa para empresas que já utilizam a infraestrutura da Microsoft. Para utilizar o Azure, você precisa configurar a conta e chave de acesso no portal da Azure. A integração com Python é feita usando a biblioteca azure-cognitiveservices-speech.



### Comparando modelos exemplo

```
import litellm
import os
from litellm import batch_completion_models

# Configurando as chaves de API
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Fazendo a chamada para comparar as respostas
response = batch_completion_models(
    models=["groq/gemma2-9b-it",
            "groq/llama3-groq-70b-8192-tool-use-preview"],
    messages=[{"role": "user", "content": "Hey, how's it going"}]
)

# Exibindo os resultados
print(response)
```



### Extraindo embbedings pela biblioteca spacy exemplo

```
import spacy

# Carregar o modelo em português
nlp = spacy.load("pt_core_news_md")

# Lista de textos
data = [
    {
        "id": "occurrence1",
        "text": "Ouro presente em veios de quartzo em uma formação hidrotermal. Observa-se alta concentração de ouro em zonas de fraturas e falhas."
    },
    {
        "id": "occurrence2",
        "text": "Associação do ouro com sulfetos, especialmente pirita e arsenopirita, em ambiente de rochas metavulcânicas. Indica potencial para depósito orogênico de ouro."
    },
    {
        "id": "occurrence3",
        "text": "Ouro aluvial encontrado em depósitos de cascalho próximo a rios e córregos. Indica transporte e concentração secundária de ouro."
    },
    {
        "id": "occurrence4",
        "text": "Presença de ouro em formações de skarn associadas a intrusões ígneas graníticas. Indica formação relacionada a processos de metamorfismo de contato."
    },
    {
        "id": "occurrence5",
        "text": "Ouro disseminado em formações sedimentares de origem marinha, em conglomerados ricos em minerais pesados. Potencial para depósito do tipo placer ou paleoplacer."
    },
]

# Criar embeddings para cada ocorrência
embeddings = []
for occurrence in data:
    doc = nlp(occurrence["text"])
    embeddings.append({"id": occurrence["id"], "embedding": doc.vector})

# Resultados
for emb in embeddings:
    print(f"ID: {emb['id']}, Embedding: {emb['embedding'][:5]}...")  # Mostrando uma parte do embedding para simplificação
```



## Pontos importantes

- Interagir com modelos de linguagem de grandes dimensões (LLMs) utilizando a biblioteca litellm e a API GROQ;
- Criar um chatbot simples que se comunica com um modelo de LLM, armazenando o histórico da conversa para manter o contexto;
- Utilizar diferentes modelos de LLMs, como gemma2-9b-it e llama3-groq-70b-8192-tool-use-preview, para obter diferentes tipos de respostas;
- Obter informações do mundo real através de uma API externa, neste caso, a API de previsão do tempo OpenWeatherMap;
- Criar um dicionário para definir as ferramentas que o chatbot pode utilizar;
- Atualizar a função de chamada da API para integrar as ferramentas;
- Adicionar múltiplas ferramentas ao chatbot, incluindo APIs e funções que manipulam dados;
- Utilizar a biblioteca Pandas para carregar dados de um arquivo CSV e convertê-los para texto para o chatbot;
- Integrar ferramentas que usam APIs externas, arquivos CSV e outras bibliotecas como Pandas ao seu chatbot;
- Como utilizar a biblioteca Pinecone para fazer o embedding de textos e salvá-los em uma base de dados vetorial.
- Processo de carregar um livro em PDF para a base de conhecimento, dividindo-o em chunks de texto e realizando o embedding de cada um deles.
- Criar uma função para consultar uma base de dados vetorial e retornar o trecho de texto mais semelhante à pergunta do usuário;
- Integrar a função de consulta à base de dados ao chatbot, utilizando uma palavra-chave para determinar quando a consulta deve ser realizada;
- Adaptar a função de chatbot para atender as necessidades do Gradio, utilizando uma função response que recebe mensagens e histórico como entrada;
- Criar uma interface de chat com o Gradio, utilizando a função response para gerar as respostas do chatbot;
- Implementar a função ChatInterface do Gradio para criar a interface do chat, personalizando o título e o input do usuário.
- Criar uma segunda aba na sua aplicação utilizando o Gradio, adicionando funcionalidade de interação por voz;
- Utilizar o modelo Whisper da OpenAI para transcrever áudio em tempo real, utilizando a biblioteca Transformers;
- Subir a aplicação para o HuggingFace Spaces, tornando-a disponível para outros usuários;
- Adaptar o código para acessar as chaves de API diretamente do sistema, em vez de armazená-las no colab, garantindo segurança e acessibilidade;

### APIs e dados utilizados:

- https://openweathermap.org/api

acesso à previsão do tempo real-time

- https://console.groq.com/playground

acesso ao modelo llama

- https://app.pinecone.io/organizations

banco cloud dos embbedings

- https://services.swpc.noaa.gov/products/noaa-planetary-k-index.json

alerta de tempestade solar

- https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/4.5_day.csv

extrai dados de sismos

- livro.pdf sobre geologia